In [1]:
def kv_kb_per_token(
    layers=28,
    kv_heads=2,
    head_dim=128,
    dbytes=2,
):
    return 2 * layers * kv_heads * head_dim * dbytes / 1024

KB_PER_TOKEN = kv_kb_per_token()
BLOCK_TOKENS = 16
BLOCK_KB = BLOCK_TOKENS * KB_PER_TOKEN

print("KB per token:", KB_PER_TOKEN)
print("KB per block:", BLOCK_KB)

KB per token: 28.0
KB per block: 448.0


In [2]:
class SlabAllocator:
    def __init__(self, budget_kb, max_len=4096):
        self.budget_kb = budget_kb
        self.max_len = max_len
        self.used_kb = 0
        self.resident = {}

    def admit(self, seq_id):
        need = self.max_len * KB_PER_TOKEN

        if self.used_kb + need > self.budget_kb:
            return False

        self.used_kb += need
        self.resident[seq_id] = need
        return True

    def complete(self, seq_id):
        self.used_kb -= self.resident.pop(seq_id)

print("Slab allocator is ready")

Slab allocator is ready


In [3]:
class BlockPoolAllocator:
    def __init__(self, budget_kb, block_kb=BLOCK_KB):
        self.block_kb = block_kb
        self.total_blocks = int(budget_kb // block_kb)
        self.free_blocks = self.total_blocks
        self.block_tables = {}

    def admit(self, seq_id):
        if self.free_blocks < 1:
            return False

        self.free_blocks -= 1
        self.block_tables[seq_id] = 1
        return True

    def grow(self, seq_id, current_len_tokens):
        needed_blocks = -(-current_len_tokens // BLOCK_TOKENS)
        held = self.block_tables[seq_id]

        if needed_blocks > held:
            extra = needed_blocks - held

            if self.free_blocks < extra:
                return False

            self.free_blocks -= extra
            self.block_tables[seq_id] = needed_blocks

        return True

    def complete(self, seq_id):
        self.free_blocks += self.block_tables.pop(seq_id)

print("Block pool allocator is ready")

Block pool allocator is ready


In [4]:
import random

random.seed(7)

def make_workload(n_sequences=60, max_len=4096):
    lengths = []

    for _ in range(n_sequences):
        if random.random() < 0.85:
            lengths.append(random.randint(50, 400))
        else:
            lengths.append(random.randint(2000, max_len))

    return lengths

WORKLOAD = make_workload()

print("Number of sequences:", len(WORKLOAD))
print("First 10 lengths:", WORKLOAD[:10])

Number of sequences: 60
First 10 lengths: [127, 74, 324, 348, 309, 94, 85, 332, 339, 164]


In [5]:
BUDGET_KB = 2 * 1024 * 1024

def simulate_slab(workload):
    allocator = SlabAllocator(BUDGET_KB)
    admitted = 0
    rejected = 0

    for seq_id, length in enumerate(workload):
        if allocator.admit(seq_id):
            admitted += 1
        else:
            rejected += 1

    return {
        "admitted": admitted,
        "rejected": rejected,
        "resident": len(allocator.resident),
        "used_kb": allocator.used_kb,
    }

def simulate_blockpool(workload):
    allocator = BlockPoolAllocator(BUDGET_KB)
    admitted = 0
    rejected = 0

    for seq_id, length in enumerate(workload):
        if not allocator.admit(seq_id):
            rejected += 1
            continue

        if allocator.grow(seq_id, length):
            admitted += 1
        else:
            allocator.complete(seq_id)
            rejected += 1

    used_blocks = allocator.total_blocks - allocator.free_blocks

    return {
        "admitted": admitted,
        "rejected": rejected,
        "resident": len(allocator.block_tables),
        "used_blocks": used_blocks,
        "used_kb": used_blocks * BLOCK_KB,
    }

slab_result = simulate_slab(WORKLOAD)
blockpool_result = simulate_blockpool(WORKLOAD)

print("Slab:", slab_result)
print("Block pool:", blockpool_result)

Slab: {'admitted': 18, 'rejected': 42, 'resident': 18, 'used_kb': 2064384.0}
Block pool: {'admitted': 60, 'rejected': 0, 'resident': 60, 'used_blocks': 1695, 'used_kb': 759360.0}


In [6]:
import json

blockpool_advantage = round(
    blockpool_result["resident"] / slab_result["resident"],
    2,
)

report = {
    "kb_per_token": KB_PER_TOKEN,
    "block_tokens": BLOCK_TOKENS,
    "budget_kb": BUDGET_KB,
    "workload_mean_len": round(sum(WORKLOAD) / len(WORKLOAD), 2),
    "slab": slab_result,
    "blockpool": blockpool_result,
    "blockpool_advantage": blockpool_advantage,
}

with open("kv_sim_report.json", "w") as f:
    json.dump(report, f, indent=2)

print(json.dumps(report, indent=2))

{
  "kb_per_token": 28.0,
  "block_tokens": 16,
  "budget_kb": 2097152,
  "workload_mean_len": 444.4,
  "slab": {
    "admitted": 18,
    "rejected": 42,
    "resident": 18,
    "used_kb": 2064384.0
  },
  "blockpool": {
    "admitted": 60,
    "rejected": 0,
    "resident": 60,
    "used_blocks": 1695,
    "used_kb": 759360.0
  },
  "blockpool_advantage": 3.33
}


In [8]:
from google.colab import files

files.download("kv_sim_report.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:
import json
import os

REPORT_FILE = "kv_sim_report.json"

assert os.path.exists(REPORT_FILE), "kv_sim_report.json not found"

with open(REPORT_FILE) as f:
    data = json.load(f)

required_keys = [
    "kb_per_token",
    "block_tokens",
    "budget_kb",
    "workload_mean_len",
    "slab",
    "blockpool",
    "blockpool_advantage",
]

for key in required_keys:
    assert key in data, f"Missing key: {key}"

assert data["kb_per_token"] == 28.0
assert data["block_tokens"] == 16
assert data["slab"]["resident"] == 18
assert data["slab"]["rejected"] == 42
assert data["blockpool"]["resident"] == 60
assert data["blockpool"]["rejected"] == 0
assert round(data["blockpool_advantage"], 2) == 3.33

print(
    "reference: slab 18 resident, "
    "block-pool 60 resident, advantage 3.33x"
)
print("GREEN CHECK: PASS")

reference: slab 18 resident, block-pool 60 resident, advantage 3.33x
GREEN CHECK: PASS
